In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("spark://spark-master:7077") \
    .appName("KafkaTest") \
    .config("spark.driver.host", "jupyter") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config(
        "spark.jars",
        "/opt/spark/jars/spark-sql-kafka-0-10_2.12-3.5.1.jar,"
        "/opt/spark/jars/spark-token-provider-kafka-0-10_2.12-3.5.1.jar,"
        "/opt/spark/jars/kafka-clients-3.5.1.jar"
    ) \
    .enableHiveSupport() \
    .getOrCreate()

26/06/22 15:29:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [2]:
df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "students") \
    .load()

df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [3]:
parsed_df = df.selectExpr(
    "CAST(key AS STRING) as key",
    "CAST(value AS STRING) as value",
    "topic",
    "partition",
    "offset",
    "timestamp"
)

query = parsed_df.writeStream \
    .format("console") \
    .outputMode("append") \
    .start()

26/06/22 15:22:46 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-2c81de5e-f7de-4ffb-a6d9-2a0d19fd8000. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/06/22 15:22:46 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/22 15:22:50 WARN ClientUtils: Couldn't resolve server kafka:9092 from bootstrap.servers as DNS resolution failed for kafka
26/06/22 15:22:50 WARN KafkaOffsetReaderAdmin: Error in attempt 1 getting Kafka offsets: 
org.apache.kafka.common.KafkaException: Failed to create new KafkaAdminClient
	at org.apache.kafka.clients.admin.KafkaAdminClient.createInternal(KafkaAdminClient.java:551)
	at org.apache.kafka.clients.admin.Admin.create(Admin.java:144)
	at org.apache.s

In [4]:
print(dict(spark.sparkContext.getConf().getAll()))

{'spark.app.startTime': '1782141724214', 'spark.driver.extraJavaOptions': '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED -Djdk.reflect.useDirectMethodHandle=false', 'spark.driver.host': 'jupyter', 'spark.executor.id': 'driver', 'spa

In [5]:
spark.jars

AttributeError: 'SparkSession' object has no attribute 'jars'

In [6]:
spark.jars.packages


AttributeError: 'SparkSession' object has no attribute 'jars'

In [2]:
print(
    spark.sparkContext.getConf().get("spark.jars")
)

/opt/spark/jars/spark-sql-kafka-0-10_2.12-3.5.1.jar,/opt/spark/jars/spark-token-provider-kafka-0-10_2.12-3.5.1.jar,/opt/spark/jars/kafka-clients-3.5.1.jar


In [3]:
df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "students") \
    .load()

df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [4]:
parsed_df = df.selectExpr(
    "CAST(key AS STRING)",
    "CAST(value AS STRING)",
    "topic",
    "partition",
    "offset",
    "timestamp"
)

query = parsed_df.writeStream \
    .format("console") \
    .outputMode("append") \
    .start()

26/06/22 15:31:06 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-8d0d839f-07b7-4bfa-9aa9-5f40adcc46de. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/06/22 15:31:06 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/22 15:31:10 WARN ClientUtils: Couldn't resolve server kafka:9092 from bootstrap.servers as DNS resolution failed for kafka
26/06/22 15:31:10 WARN KafkaOffsetReaderAdmin: Error in attempt 1 getting Kafka offsets: 
org.apache.kafka.common.KafkaException: Failed to create new KafkaAdminClient
	at org.apache.kafka.clients.admin.KafkaAdminClient.createInternal(KafkaAdminClient.java:551)
	at org.apache.kafka.clients.admin.Admin.create(Admin.java:144)
	at org.apache.s

In [5]:
spark.sparkContext.master

'spark://spark-master:7077'

In [6]:
spark.sparkContext._jsc.sc().listJars()

JavaObject id=o91

In [7]:
print(list(spark.sparkContext._jsc.sc().listJars()))

TypeError: 'JavaObject' object is not iterable

In [8]:
dict(spark.sparkContext.getConf().getAll())

{'spark.driver.extraJavaOptions': '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED -Djdk.reflect.useDirectMethodHandle=false',
 'spark.driver.host': 'jupyter',
 'spark.driver.port': '44053',
 'spark.executor.id': 'driver',
 'spark.app

In [9]:
spark.conf.get("spark.jars", "NOT_FOUND")

'/opt/spark/jars/spark-sql-kafka-0-10_2.12-3.5.1.jar,/opt/spark/jars/spark-token-provider-kafka-0-10_2.12-3.5.1.jar,/opt/spark/jars/kafka-clients-3.5.1.jar'

In [10]:
for jar in spark.sparkContext._jsc.sc().listJars():
    print(jar)

TypeError: 'JavaObject' object is not iterable

In [11]:
from pyspark.sql import SparkSession

spark.range(1).show()

+---+
| id|
+---+
|  0|
+---+



In [12]:
spark.sparkContext.parallelize([1]).map(
    lambda x: __import__("socket").gethostbyname("kafka")
).collect()

['172.20.0.2']

In [13]:
df.explain(True)

== Parsed Logical Plan ==
StreamingRelationV2 org.apache.spark.sql.kafka010.KafkaSourceProvider@8733797, kafka, org.apache.spark.sql.kafka010.KafkaSourceProvider$KafkaTable@4b12adf2, [kafka.bootstrap.servers=kafka:9092, subscribe=students], [key#7, value#8, topic#9, partition#10, offset#11L, timestamp#12, timestampType#13], StreamingRelation DataSource(org.apache.spark.sql.SparkSession@77aa98a7,kafka,List(),None,List(),None,Map(kafka.bootstrap.servers -> kafka:9092, subscribe -> students),None), kafka, [key#0, value#1, topic#2, partition#3, offset#4L, timestamp#5, timestampType#6]

== Analyzed Logical Plan ==
key: binary, value: binary, topic: string, partition: int, offset: bigint, timestamp: timestamp, timestampType: int
StreamingRelationV2 org.apache.spark.sql.kafka010.KafkaSourceProvider@8733797, kafka, org.apache.spark.sql.kafka010.KafkaSourceProvider$KafkaTable@4b12adf2, [kafka.bootstrap.servers=kafka:9092, subscribe=students], [key#7, value#8, topic#9, partition#10, offset#11L, 